# Kaggle credentials

In [ ]:
username = "username" # @param {"type":"string","placeholder":"Kaggle username"}
key = "KGAT_xxxxxxxxxxxxxxxxxxxxxxx" # @param {"type":"string","placeholder":"Kaggle API key"}

# Initialization

## Download and unzip dataset

In [ ]:
import os
os.environ['KAGGLE_USERNAME'] = username
os.environ['KAGGLE_KEY'] = key

DATASET_VERSION = "6"
DATASET_DIR = "new-york-times-articles-comments-2020"
os.makedirs(DATASET_DIR, exist_ok=True)

import kagglehub
kagglehub.dataset_download(f'benjaminawd/new-york-times-articles-comments-2020/versions/{DATASET_VERSION}', output_dir=DATASET_DIR);

# Same behaviour of the code above but uses kaggle cli, but  cannot specify the dataset version
# !mkdir -p new-york-times-articles-comments-2020/
# !kaggle datasets download benjaminawd/new-york-times-articles-comments-2020 -p new-york-times-articles-comments-2020/
# !cd new-york-times-articles-comments-2020/ && unzip new-york-times-articles-comments-2020.zip
# !rm new-york-times-articles-comments-2020/new-york-times-articles-comments-2020.zip

## PySpark setup


In [ ]:
import os

!apt-get update -qq > /dev/null
!apt-get install -y openjdk-17-jdk-headless -qq > /dev/null
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
!pip install -q pyspark==4.0.0

## Custom modules

In [ ]:
#clone src and tests folders form the project repo
!rm -rf src tests/
!git clone --depth 1 https://github.com/fil-noct/find-similar-items-on-massive-datasets.git temp_repo
!mv temp_repo/src ./src
!mv temp_repo/tests ./tests
!rm -rf temp_repo

#run custom modules tests
!python -m pytest ./tests --import-mode=importlib
import importlib
import src.shingler
import src.minhash
import src.lsh
import src.utils.hash

importlib.reload(src.shingler)
importlib.reload(src.minhash)
importlib.reload(src.lsh)
importlib.reload(src.utils.hash)
print("Successfully imported al the modules")

In [ ]:
#CUSTOM MODULES DEMONSTRATION

from src.shingler import Shingler, ShinglerConfiguration
from src.utils.hash import get_prime_above_bucket_size_in_bytes, DynamicShingleHasher
from src.minhash import HashPermutator, MinhashGenerator, HashPermutatorConfiguration
from src.lsh import LSHConfiguration, LocalitySensitiveHasher

text1 = "lorem ipsum dolor sit amet original"
text2 = "lorem ipsum dolor sit amet different"

print("CUSTOM MODULE SHOWOFF")
print("text1: ", text1)
print("text2: ", text2)

shingles_bucket_size_in_byte=2
shingle_hasher = DynamicShingleHasher(n_bytes=shingles_bucket_size_in_byte)

#Shingling
shingler = Shingler(
    config=ShinglerConfiguration(
      type='word',
      k=3,
      hasher=shingle_hasher
    )
)
set1 = set(shingler.get_shingles(text1))
set2 = set(shingler.get_shingles(text2))

union_len = len(set1 | set2)
js = float(len(set1 & set2) / union_len) if union_len > 0 else 0.0
print(f"Jaccard similarity: {js}")

print("Readable shingles:")
print("\ttext1:", shingler.get_shingles(text1))
print("\ttext2:", shingler.get_shingles(text2))

hashed_shingles1 = shingler.get_hashed_shingles(text1)
hashed_shingles2 = shingler.get_hashed_shingles(text2)

print(f"Hashed shingles in bucket of {shingles_bucket_size_in_byte} bytes")
print("\ttext1:", hashed_shingles1)
print("\ttext2:", hashed_shingles2)

#Minhash
n_hashes = 100
mod_factor = get_prime_above_bucket_size_in_bytes(shingles_bucket_size_in_byte)
permutator = HashPermutator(
    config=HashPermutatorConfiguration(
        n=n_hashes,
        mod_factor=mod_factor
    )
)
sig1 = MinhashGenerator.get_signature(permutator, hashed_shingles1)
sig2 = MinhashGenerator.get_signature(permutator, hashed_shingles2)

print("Minhash signatures")
print("\ttext1:", sig1)
print("\ttext2:", sig2)

#LSH
B=25
R=4
lsh = LocalitySensitiveHasher(
    config=LSHConfiguration(
        signature_size=n_hashes,
        b=B,
        r=R,
        signature_item_size_in_bytes=shingles_bucket_size_in_byte,
        lsh_band_bucket_size_in_bytes=4
    )
)

print(f"LSH threshold: {lsh.get_estimated_threshold()}")
band_buckets1 = lsh.hashed_bands(sig1)
band_buckets2 = lsh.hashed_bands(sig2)
print("LSH buckets")
print("\ttext1:", band_buckets1)
print("\ttext2:", band_buckets2)
matches = list(set(band_buckets1) & set(band_buckets2))
print("LSH band, bucket collision")
print(matches)



# Find similar items pairs

## Utils functions declaration

### Pipeline context generator

In [ ]:
#util function to generate full configuration context
#params validation using configuration classes
#this generated context will be brodcasted to the whole cluster (NB all objecs are pickable)

from src.shingler import Shingler, ShinglerConfiguration, ShinglerType
from src.minhash import MinhashGenerator, HashPermutator, HashPermutatorConfiguration
from src.lsh import LocalitySensitiveHasher, LSHConfiguration
from src.utils.hash import get_required_bytes_to_store_int

from typing import Callable

def generate_context(
    shingler_type: ShinglerType,
    shingle_grams: int,
    n_minhash_functions: int,
    minhash_modfactor: int,
    bands: int,
    rows: int,
    str_sanitizer: Callable[[str], str] | None = None,
    shingle_hasher:  Callable[[str], int] | None = None,
    ):

  shingler = Shingler(
      config=ShinglerConfiguration(
          type=shingler_type,
          k=shingle_grams,
          sanitizer=str_sanitizer,
          hasher=shingle_hasher
      )
  )

  permutator = HashPermutator(
      config=HashPermutatorConfiguration(
          n=n_minhash_functions,
          mod_factor=minhash_modfactor
      )
  )

  lsh = LocalitySensitiveHasher(
      config=LSHConfiguration(
          signature_size=n_minhash_functions,
          b=bands,
          r=rows,
          signature_item_size_in_bytes=get_required_bytes_to_store_int(minhash_modfactor),
          lsh_band_bucket_size_in_bytes=4
      )
  )
  return {"shingler": shingler, "permutator": permutator, "lsh": lsh}

### Plot greedy search

In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

def plot_lsh_s_curves(solutions, threshold, error, max_signature_len):
    selected_sols = solutions[:3]
    fig, ax = plt.subplots(figsize=(8, 6))

    s = np.linspace(0, 1, 500)
    ax.axvspan(
        max(0, threshold - error),
        min(1, threshold + error),
        color="red",
        alpha=0.08,
        label=f"Tolerance (±{error:.2f})",
    )
    ax.axvline(x=threshold, color='black', linestyle='--', linewidth=1.5, alpha=0.7, label=f"Target ({threshold:.2f})")

    for idx, sol in enumerate(selected_sols):
        b, r, n, t_real, j_error = sol['b'], sol['r'], sol['n'], sol['t_real'], sol["jaccard_error"]
        p_candidate = 1 - (1 - s**r)**b
        curve_label = f"$b={b}, r={r}$ ($n={n}$, $t={t_real:.2f}$ $Jerror={j_error}$)"
        ax.plot(s, p_candidate, label=curve_label, linewidth=2.5)
        ax.plot(t_real, 0.5, marker='o', markersize=6, markeredgecolor='white')

    ax.axhline(y=0.5, color='gray', linestyle=':', alpha=0.5, linewidth=1)
    ax.set_title("Locality Sensitive Hashing (LSH) $S$-Curve Profiles", fontsize=13, fontweight='bold', pad=12)
    ax.set_xlabel("Jaccard Similarity ($s$)", fontsize=11, fontweight='semibold')
    ax.set_ylabel("Probability of Candidate Pair $P(s)$", fontsize=11, fontweight='semibold')

    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1.02)

    ax.legend(
        title="LSH Parameters",
        title_fontsize=10,
        loc='upper left',
        frameon=True,
        facecolor='white',
        framealpha=0.9,
        fontsize=9.5
    )

    sns.despine()
    plt.tight_layout()
    plt.show()

### Jaccard similarity on sets

In [ ]:
from pyspark.context import SparkContext

def calculate_jaccard(broadcast_ctx: SparkContext, text1:str, text2:str):
    shingler: Shingler = broadcast_ctx.value["shingler"]
    set1 = set(shingler.get_shingles(text1))
    set2 = set(shingler.get_shingles(text2))

    if not set1 or not set2:
        return 0.0

    union_len = len(set1 | set2)
    return float(len(set1 & set2) / union_len) if union_len > 0 else 0.0

## PySpark implementation

### Dataset params

In [ ]:
CSV_PATH = "new-york-times-articles-comments-2020/nyt-articles-2020.csv" # @param {type:"string", placeholder: "csv path"}
CSV_LIMIT = 2500 # @param {type:"integer", placeholder: "CSV lines limit, leave empty to use the whole file"}

### Hyper params

In [ ]:
K = 3 # @param {type:"integer", placeholder: "k-grams shingle"}
SHINGLER_TYPE = "word" # @param ["char", "word"]
SHINGLER_BUCKET_SIZE_IN_BYTES=8 # @param {type:"integer", placeholder: "Byte for the hashed shingle bucket"}

In [ ]:
OPTIMIZATION_OBJECTIVE = "precision" # @param ["precision", "compression"]
THRESHOLD = 0.7 # @param {"type":"slider","min":0,"max":1,"step":0.01}
ERROR = 0.1 # @param {type:"number", placeholder: "error"}
MAX_SIGNATURE_LEN = 500 # @param {type:"integer", placeholder: "max signature length"}

### LSH params tuning based on hyper params

In [ ]:
import math
from typing import Literal

# grid param search for lsh
# starts from the similarity threshold target, absolute error and optimization objective
# base on hyperparams this function may not find any solution
def grid_param_search_for_lsh(threshold, max_signature_len, optimization_objective:Literal["precision", "compression"], error=0.10):
    n_min = int(math.ceil(1.0 / (error ** 2)))
    r_absolute_limit = max_signature_len // 2
    r_max = 2

    while r_max <= r_absolute_limit:
        b_ideal = (1.0 / threshold) ** r_max
        n_estimated = b_ideal * r_max
        if n_estimated > max_signature_len:
            r_max = r_max - 1
            break
        r_max += 1

    r_max = min(r_max, r_absolute_limit)

    valid_solutions = []

    for r in range(2, r_max+1):
        b_ideal = (1.0 / threshold) ** r
        b = round(b_ideal)

        if b < 2:
            continue

        n = b * r

        if n >= n_min:
            t = (1.0 / b) ** (1.0 / r)
            t_error = abs(t - threshold)

            valid_solutions.append({
                "n": n,
                "b": b,
                "r": r,
                "t_real": round(t, 4),
                "t_error": round(t_error, 4),
                "jaccard_error": round(1.0 / math.sqrt(n), 4)
            })

    if not valid_solutions:
        raise ValueError(f"No combination fulfill the desired threshold ({threshold}) with the specified error ({error})")

    if optimization_objective == "compression":
        valid_solutions.sort(key=lambda x: (x["n"], x["t_error"]))
    elif optimization_objective == "precision":
        valid_solutions.sort(key=lambda x: (x["jaccard_error"], x["t_error"]))
    else:
        raise ValueError(f"Invalid optimization objective: {optimization_objective}")

    return valid_solutions

In [ ]:
#PARAMS INITIALIZATION
from src.utils.hash import DynamicShingleHasher, get_prime_above_bucket_size_in_bytes
from src.utils.text import keep_letters_lowercase

shingle_hasher = DynamicShingleHasher(n_bytes=SHINGLER_BUCKET_SIZE_IN_BYTES)
mod_factor = get_prime_above_bucket_size_in_bytes(SHINGLER_BUCKET_SIZE_IN_BYTES)

#LSH hyper params
solutions = grid_param_search_for_lsh(
    optimization_objective=OPTIMIZATION_OBJECTIVE,
    threshold=THRESHOLD,
    max_signature_len=MAX_SIGNATURE_LEN,
    error=ERROR
)
best_solution = solutions[0]
N = best_solution["n"]
B = best_solution["b"]
R = best_solution["r"]

print("\nSIMILARITY THRESHOLD:")
print(f"\tthreshold: {THRESHOLD}")
print(f"\terror: {ERROR}")
print("\nCLUSTER CONTEXT:")
print("\tShingles:")
print(f"\t\tk: {K}")
print(f"\t\ttype: {SHINGLER_TYPE}")
print(f"\t\thashed shingle bucket size in bytes: {SHINGLER_BUCKET_SIZE_IN_BYTES}")
print("\tMinhash:")
print(f"\t\tsignature length: {best_solution["n"]}")
print("\tLSH:")
print(f"\t\tbands: {best_solution["b"]}")
print(f"\t\trows: {best_solution["r"]}")
print("\nLSH THRESHOLD CURVE")
plot_lsh_s_curves(solutions=solutions, threshold=THRESHOLD, error=ERROR, max_signature_len=MAX_SIGNATURE_LEN)


### PySpark Map Reduce job

In [ ]:
#CLUSTER INITIALIZATION
from pyspark.sql import SparkSession, Row

#simulate spark cluster using multi thea
spark = SparkSession.builder \
        .appName("SimilarityMapReduce") \
        .master("local[*]") \
        .getOrCreate()

cluster_context = generate_context(
    shingler_type=SHINGLER_TYPE,
    shingle_grams=K,
    n_minhash_functions=N,
    minhash_modfactor=mod_factor,
    bands=B,
    rows=R,
    str_sanitizer=keep_letters_lowercase,
    shingle_hasher= shingle_hasher,
)

broadcast_ctx = spark.sparkContext.broadcast(cluster_context)

In [ ]:
# function to convert csv data frame to a key value tuple
# this function can be changed based on the dataframe type and returns tuples like (id,text)
def df_to_tuple(row):
    doc_id = row["uniqueID"]
    abstract = row["abstract"]

    if doc_id is None or str(doc_id).strip() == "" or not doc_id.startswith("nyt://") :
        return []
    if abstract is None or not isinstance(abstract, str) or abstract.strip() == "":
        return []

    return[ (str(doc_id).strip(), abstract.strip())]

def document_to_hashed_bands(content: str):
    shingler: Shingler = broadcast_ctx.value["shingler"]
    permutator: HashPermutator = broadcast_ctx.value["permutator"]
    lsh: LocalitySensitiveHasher = broadcast_ctx.value["lsh"]

    hashed_shingles = shingler.get_hashed_shingles(content)
    if len(hashed_shingles) == 0:
        return []
    signature = MinhashGenerator.get_signature(permutator, hashed_shingles)
    bucket_ids = lsh.hashed_bands(signature)
    return bucket_ids

print("STARTING PYSPARK JOB")
print("Looking for candidate pairs across...")

df = spark.read.csv(header=True, inferSchema=True, path=CSV_PATH)

if CSV_LIMIT is not None:
    df = df.limit(CSV_LIMIT)

rdd = df.rdd \
      .flatMap(df_to_tuple) \
      .map(lambda row: (row[0], row[1], document_to_hashed_bands(row[1])))

buckets_with_docs = rdd.flatMap(lambda row: [((band_id, bucket_id), [(row[0], row[1])]) for band_id, bucket_id in row[2]]) \
                  .reduceByKey(lambda a,b: a+b) \
                  .filter(lambda row: len(row[1]) > 1)

candidate_pairs = buckets_with_docs \
                  .flatMap(lambda row: [(row[1][i], row[1][j], ) for i in range(len(row[1])) for j in range(i + 1, len(row[1]))]) \
                  .distinct()
print("Done.")

In [ ]:
print("Calculating jaccard similarity on candidates...")
# tuples ((id1, id2), text1, text2, jaccard_similarity)
pairs_with_jaccard = candidate_pairs \
                    .map(lambda row:((row[0][0], row[1][0]), row[0][1], row[1][1], calculate_jaccard(broadcast_ctx, row[0][1], row[1][1])))

non_identical_pairs = pairs_with_jaccard.filter(lambda row: row[3] < 1.0)
print("Done.")
print(f"All candidate pairs found: {pairs_with_jaccard.count()}")
print(f"All non identical pairs found: {non_identical_pairs.count()}")
print("First 5  non identical pairs")
sample = non_identical_pairs.take(5)
for i, s in enumerate(sample):
  print(f"Pair {i}:")
  print(f"\tID1: {s[0][0]}")
  print(f"\tID2: {s[0][1]}")
  print(f"\tText1: {s[1]}")
  print(f"\tText2: {s[2]}")
  print(f"\tJaccard similarity: {s[3]}")

In [ ]:
classified_pairs = non_identical_pairs.map(lambda row: (
    row[3] >= THRESHOLD, # True Positive
    THRESHOLD - ERROR <= row[3] < THRESHOLD, # TP in Error Bound
    row[3] < THRESHOLD - ERROR, # False Positive
    row
))

tp_count = classified_pairs.filter(lambda x: x[0]).count()
tp_in_error_count = classified_pairs.filter(lambda x: x[1]).count()
fp_pairs_rdd = classified_pairs.filter(lambda x: x[2]).map(lambda x: x[3])
fp_count = fp_pairs_rdd.count()

print(f"True Positives (TP): {tp_count}")
print(f"True Positives in Error Bound: {tp_in_error_count}")
print(f"False Positives (FP): {fp_count}")
print("="*40 + "\n")

fp_samples = fp_pairs_rdd.collect()

if fp_samples:
    for i, pair in enumerate(fp_samples, 1):
        pair_ids = pair[0]
        text1 = pair[1]
        text2 = pair[2]
        jaccard_sim = pair[3]

        print(f"\n[False Positive #{i}]")
        print(f"  ID 1               : {pair_ids[0]}")
        print(f"  ID 2               : {pair_ids[1]}")
        print(f"  Jaccard Similarity : {jaccard_sim:.4f} (Threshold: {THRESHOLD})")
        print(f"  Text 1             : {text1}")
        print(f"  Text 2             : {text2}")